In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re


In [ ]:
data_Deals= pd.read_excel('/content/drive/MyDrive/Deals (Done).xlsx')
data_Deals.head()

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,SLA,Content,...,Product,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch
0,5805028000056864695,Ben Hall,NaN,NaN,New Lead,NaN,/eng/test,03.07.23women,NaN,v16,...,NaN,NaN,21.06.2024 15:30,NaN,NaN,NaN,NaN,5.805028e+18,NaN,NaN
1,5805028000056859489,Ulysses Adams,NaN,NaN,New Lead,NaN,/at-eng,NaN,NaN,NaN,...,Web Developer,Morning,21.06.2024 15:23,6.0,NaN,0,2000,5.805028e+18,NaN,NaN
2,5805028000056832357,Ulysses Adams,21.06.2024,D - Non Target,Lost,Non target,/at-eng,engwien_AT,00:26:43,b1-at,...,NaN,NaN,21.06.2024 14:45,NaN,NaN,NaN,NaN,5.805028e+18,NaN,NaN
3,5805028000056824246,Eva Kent,21.06.2024,E - Non Qualified,Lost,Invalid number,/eng,04.07.23recentlymoved_DE,01:00:04,bloggersvideo14com,...,NaN,NaN,21.06.2024 13:32,NaN,NaN,NaN,NaN,5.805028e+18,NaN,NaN
4,5805028000056873292,Ben Hall,21.06.2024,D - Non Target,Lost,Non target,/eng,discovery_DE,00:53:12,website,...,NaN,NaN,21.06.2024 13:21,NaN,NaN,NaN,NaN,5.805028e+18,NaN,NaN


In [ ]:
data_Deals.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21593 entries, 0 to 21592
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Id                   21593 non-null  int64  
 1   Deal Owner Name      21592 non-null  object 
 2   Closing Date         14645 non-null  object 
 3   Quality              19340 non-null  object 
 4   Stage                21593 non-null  object 
 5   Lost Reason          16124 non-null  object 
 6   Page                 21593 non-null  object 
 7   Campaign             16067 non-null  object 
 8   SLA                  15533 non-null  object 
 9   Content              14147 non-null  object 
 10  Term                 12454 non-null  object 
 11  Source               21593 non-null  object 
 12  Payment Type         496 non-null    object 
 13  Product              3592 non-null   object 
 14  Education Type       3299 non-null   object 
 15  Created Time         21593 non-null 

In [ ]:
duplicates = data_Deals[data_Deals.duplicated(
    subset=['Deal Owner Name', 'Stage', 'Created Time', 'Contact Name'],
    keep=False
)]

print("Количество найденных дубликатов:", duplicates.shape[0])


Количество найденных дубликатов: 100


In [ ]:
# Посмотрим, какие у нас числовые столбцы
data_Deals.select_dtypes(include='number').columns

Index(['Id', 'Course duration', 'Months of study', 'Contact Name'], dtype='object')

In [ ]:
# Общая сводная таблица статистики
data_Deals.describe()


mean_values = data_Deals.mean(numeric_only=True)
median_values = data_Deals.median(numeric_only=True)
mode_values = data_Deals.mode().iloc[0]  # берём первую строку (мод может быть несколько)
range_values = data_Deals.max(numeric_only=True) - data_Deals.min(numeric_only=True)

print("Средние значения:\n", mean_values)
print("\nМедианы:\n", median_values)
print("\nМоды:\n", mode_values)
print("\nДиапазон (max - min):\n", range_values)

Средние значения:
 Id                 5.805028e+18
Course duration    1.019849e+01
Months of study    5.442857e+00
Contact Name       5.805028e+18
dtype: float64

Медианы:
 Id                 5.805028e+18
Course duration    1.100000e+01
Months of study    5.000000e+00
Contact Name       5.805028e+18
dtype: float64

Моды:
 Id                                   5805028000000922001
Deal Owner Name                            Charlie Davis
Closing Date                                  02.04.2024
Quality                                E - Non Qualified
Stage                                               Lost
Lost Reason                               Doesn't Answer
Page                                                /eng
Campaign               performancemax_digitalmarkt_ru_DE
SLA                                             00:10:11
Content                                  _{region_name}_
Term                                                wide
Source                                      Faceb

In [ ]:
categorical_cols = ['Quality', 'Stage', 'Source', 'Product', 'Education Type']

for col in categorical_cols:
    print(f"\nПоле: {col}")
    print(data_Deals[col].value_counts(dropna=False))
    print("Уникальных категорий:", data_Deals[col].nunique())



Поле: Quality
Quality
E - Non Qualified    7634
D - Non Target       6248
C - Low              3459
NaN                  2253
B - Medium           1564
A - High              432
F                       3
Name: count, dtype: int64
Уникальных категорий: 6

Поле: Stage
Stage
Lost                         15743
Call Delayed                  2248
Registered on Webinar         2072
Payment Done                   858
Waiting For Payment            325
Qualificated                   128
Registered on Offline Day      100
Need to Call - Sales            33
Need To Call                    31
Test Sent                       25
Need a consultation             23
New Lead                         6
Free Education                   1
Name: count, dtype: int64
Уникальных категорий: 13

Поле: Source
Source
Facebook Ads      4850
Google Ads        4226
Organic           2590
Tiktok Ads        2051
SMM               1730
Youtube Ads       1657
CRM               1656
Bloggers          1089
Telegram posts 

In [ ]:
data_Deals = data_Deals.drop_duplicates(
    subset=['Deal Owner Name', 'Stage', 'Created Time', 'Contact Name']
)


In [ ]:
#удалим одну пустую строку
data_Deals = data_Deals.dropna(subset=['Deal Owner Name'])


# меняем формат на category
data_Deals['Deal Owner Name'] = data_Deals['Deal Owner Name'].astype('category')

In [ ]:
data_Deals.isna().sum()

,0
Id,0
Deal Owner Name,0
Closing Date,6939
Quality,2248
Stage,0
Lost Reason,5463
Page,0
Campaign,5510
SLA,6034
Content,7433


Deal Owner Name было 30 пропусков обработаны вручную заполнялись в соответствии с CONTACTID из датасет Contacts

In [ ]:
#не заполняем пустые значения, т.к. они логичны и при заполнении могут искажать формат колонки
#меняем формат на datetime
data_Deals['Closing Date'] = pd.to_datetime(data_Deals['Closing Date'], errors='coerce')


/tmp/ipython-input-2928960436.py:3: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  data_Deals['Closing Date'] = pd.to_datetime(data_Deals['Closing Date'], errors='coerce')


In [ ]:
data_Deals['Quality'].unique()


array([nan, 'D - Non Target', 'E - Non Qualified', 'B - Medium',
       'C - Low', 'A - High', 'F'], dtype=object)

колонка Quality заменяем значения на
    'D - Non Target': 1,
    'E - Non Qualified': 2,
    'C - Low': 3,
    'B - Medium': 4,
    'A - High': 5,
    'F': 0
    nun : 0
    так будет логически понятен рейтинг менеджера и если оценки качества нет

In [ ]:
# замена
quality_map_num = {
    'D - Non Target': 1,
    'E - Non Qualified': 2,
    'C - Low': 3,
    'B - Medium': 4,
    'A - High': 5,
    'F': 0
}

# Заменяем текстовые категории на числа
data_Deals['Quality'] = data_Deals['Quality'].replace(quality_map_num)

# Пропуски тоже заменяем на 0
data_Deals['Quality'] = data_Deals['Quality'].fillna(0)

# Проверим результат
print(data_Deals['Quality'].value_counts().sort_index())



Quality
0.0    2251
1.0    6235
2.0    7603
3.0    3455
4.0    1561
5.0     432
Name: count, dtype: int64


/tmp/ipython-input-2992633528.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data_Deals['Quality'] = data_Deals['Quality'].replace(quality_map_num)


In [ ]:
data_Deals['Quality'] = data_Deals['Quality'].astype(int)


In [ ]:
data_Deals['Stage'].unique()

array(['New Lead', 'Lost', 'Need a consultation', 'Need To Call',
       'Call Delayed', 'Qualificated', 'Registered on Webinar',
       'Waiting For Payment', 'Need to Call - Sales', 'Test Sent',
       'Payment Done', 'Registered on Offline Day', 'Free Education'],
      dtype=object)

In [ ]:
data_Deals['Stage'] = data_Deals['Stage'].astype('category')

In [ ]:
data_Deals['Lost Reason'].unique()

array([nan, 'Non target', 'Invalid number', 'Duplicate', 'Inadequate',
       'Expensive', 'needs time to think', 'Not for myself',
       'Considering a different direction in IT', "Doesn't Answer",
       'Changed Decision', 'The contract did not fit',
       'Stopped Answering', 'Gutstein refusal',
       "Didn't leave an application",
       'Does not know how to use a computer',
       'Conditions are not suitable', 'Thought for free',
       'Does not speak English', 'Went to Rivals', 'Next stream',
       'Refugee'], dtype=object)

In [ ]:
data_Deals['Lost Reason'] = data_Deals['Lost Reason'].astype(str)

In [ ]:
data_Deals['Lost Reason'] = data_Deals['Lost Reason'].astype(str)
# Активные сделки без причины
data_Deals.loc[data_Deals['Lost Reason'].isna() & (data_Deals['Stage'] != 'Lost'), 'Lost Reason'] = 'Active deal'

# Проигранные сделки без причины
data_Deals.loc[data_Deals['Lost Reason'].isna() & (data_Deals['Stage'] == 'Lost'), 'Lost Reason'] = 'Lost (no reason)'


In [ ]:
#меняем формат на category
data_Deals['Lost Reason'] = data_Deals['Lost Reason'].astype('category')

In [ ]:
data_Deals['Page'].unique()

array(['/eng/test', '/at-eng', '/eng', 'eng/digital-marketing', '/',
       '/email', '/webinar', '/at-eng/digital-marketing', '/eng/ux-ui',
       '/direct', '/at-end/web-developer', 'eng/web-developer',
       '/specialoffer', '/pl-eng', '/pl-eng/web-developer',
       '/digital-marketing', '/ux-ui', '/web-developer', '/test',
       '/eng/career', '/account', '/at-ru/ux/ui',
       '/pl-eng/digital-marketing', '/pl-eng/ux-ui',
       '/at/digital-marketing', '/page', '/event', '/workshop',
       '/welcomepage', '/smm', '/ppc', '/offer', '/page1', '/course'],
      dtype=object)

In [ ]:
# Чистим от лишних символов
data_Deals['Page'] = data_Deals['Page'].str.lower().apply(
    lambda x: '/' + x if pd.notna(x) and not x.startswith('/') else x
)


In [ ]:
data_Deals['Page'] = data_Deals['Page'].replace({
    '/at-end/web-developer': '/at-eng/web-developer',
    '/at-ru/ux/ui': '/at-ru/ux-ui'
})


In [ ]:
print(sorted(data_Deals['Page'].unique()))

['/', '/account', '/at-eng', '/at-eng/digital-marketing', '/at-eng/web-developer', '/at-ru/ux-ui', '/at/digital-marketing', '/course', '/digital-marketing', '/direct', '/email', '/eng', '/eng/career', '/eng/digital-marketing', '/eng/test', '/eng/ux-ui', '/eng/web-developer', '/event', '/offer', '/page', '/page1', '/pl-eng', '/pl-eng/digital-marketing', '/pl-eng/ux-ui', '/pl-eng/web-developer', '/ppc', '/smm', '/specialoffer', '/test', '/ux-ui', '/web-developer', '/webinar', '/welcomepage', '/workshop']


In [ ]:
count_root = (data_Deals['Page'] == '/').sum()
count_root

np.int64(1075)

In [ ]:
# Скорее всего пользователь пришел с главной страницы, поэтому переименуем
data_Deals['Page'] = data_Deals['Page'].replace({'/': '/main'})


data_Deals['Page'] = data_Deals['Page'].astype('category')

In [ ]:
data_Deals['Campaign'].unique()

array(['03.07.23women', nan, 'engwien_AT', '04.07.23recentlymoved_DE',
       'discovery_DE', 'youtube_shorts_DE', 'brand_search_eng_DE',
       '1406start', '20.05.24interests_DE', 'performancemax_eng_DE',
       '12.07.2023wide_DE', '1006start', '24.09.23retargeting_DE',
       'germany_DE', 'performancemax_wide_AT', '07.07.23LAL_DE',
       'webinar1906', 'germania_DE', '02.07.23wide_DE',
       '22.05.2024wide_DE', 'blog_DE', 'blog2_DE', '17.03.24wide_AT',
       '12.06.24wide_DE', 'Jobs_germany_DE', '2005_Lost_DE', 'uk_DE',
       '08.04.24wide_webinar_DE', '08.06.24wide_webinar_DE', 'Akademia',
       'BloggerIvan', 'Genie_DE', 'Live_DE', '1706_DE',
       'performancemax_digitalmarkt_ru_DE', '12.09.23interests_Uxui_DE',
       '5555_DE', 'ASA_de_DE', '2905start', 'webinar1604',
       'bloggerfrai_DE', 'bloggerdr_DE', 'Trigger_DE', 'Bloggerel_DE',
       'Forum_DE', 'Consult_DE', 'work_DE', 'Bolgspeak_DE', 'tyk_DE',
       'of_DE', 'Berlin_DE', 'Markt_DE', 'BloggerShina_DE',
   

In [ ]:

#заполняем пустые значения
data_Deals['Campaign'] = data_Deals['Campaign'].fillna('unknown')

#меняем формат
data_Deals['Campaign'] = data_Deals['Campaign'].astype('category')

In [ ]:
# Преобразуем всё в строки (чтобы NaN стали строками None)
data_Deals['SLA'] = data_Deals['SLA'].astype(str)

# Преобразуем в timedelta, ошибки превратятся в NaT
data_Deals['SLA'] = pd.to_timedelta(data_Deals['SLA'], errors='coerce')

In [ ]:
data_Deals['SLA'] = pd.to_timedelta(data_Deals['SLA'], errors='coerce')

# Преобразуем в часы для удобства анализа
data_Deals['SLA'] = data_Deals['SLA'].dt.total_seconds() / 3600

# Базовая описательная статистика
data_Deals['SLA'].describe()

,SLA
count,15503.000000
mean,32.209850
std,204.987300
min,0.000833
25%,1.212917
50%,5.524167
75%,15.637083
max,7474.573333


In [ ]:
# Преобразуем в timedelta, ошибки превратятся в NaT
#data_Deals['SLA'] = pd.to_timedelta(data_Deals['SLA'], errors='coerce')

In [ ]:
data_Deals['Content'].unique()

array(['v16', nan, 'b1-at', 'bloggersvideo14com', 'website',
       'bloggersvideo2june', '152789402780_{region_name}_695563281558',
       'v15', '_{region_name}_', 'bloggersvideo16com', 'b9',
       'search_terms', 'bloggersvideo9com', 'b4python-developer',
       'bloggersvideo11', 'Audience', 'bloggersvideo10',
       'bloggersvideo23com', 'bloggersvideo25com', 'bloggersvideo12com',
       '152789402780_{region_name}_668024583824', 'bloggersvideo24com',
       'bloggersvideo16com_at', 'bloggersjune17', 'bloggersvideo1june',
       'b0', 'bloggersvideo15com',
       '151836595805_{region_name}_699672039100', 'v7webinar',
       'bloggersvideo18webinar', 'v6webinar',
       '151836595805_{region_name}_699672039103', 'bloggersvideo1webinar',
       '151836595805_{region_name}_699672039109', 'bloggersvideo2webinar',
       'bloggersvideo18com', 'bloggersvideo12com_at',
       '151836595805_{region_name}_699672039106', 'bloggersvideo19com',
       '151836595805_{region_name}_67380133699

In [ ]:
import re


import re
import pandas as pd

def clean_content(value):
    #  Пропуски
    if pd.isna(value) or str(value).strip() == "":
        return "unknown_content"

    # Приведение к нижнему регистру
    value = str(value).lower().strip()

    value = re.sub(r"\{.*?\}", "", value)
    value = re.sub(r"[^a-z0-9\-_]", "", value)
    value = re.sub(r"[-_]+", "_", value).strip("_")

    # Если после всего строка пустая — заменяем
    if value == "":
        value = "unknown_content"

    return value

# Применяем
data_Deals["Content"] = data_Deals["Content"].apply(clean_content)



In [ ]:
data_Deals['Content'].unique()

array(['v16', 'unknown_content', 'b1_at', 'bloggersvideo14com', 'website',
       'bloggersvideo2june', '152789402780_695563281558', 'v15',
       'bloggersvideo16com', 'b9', 'search_terms', 'bloggersvideo9com',
       'b4python_developer', 'bloggersvideo11', 'audience',
       'bloggersvideo10', 'bloggersvideo23com', 'bloggersvideo25com',
       'bloggersvideo12com', '152789402780_668024583824',
       'bloggersvideo24com', 'bloggersvideo16com_at', 'bloggersjune17',
       'bloggersvideo1june', 'b0', 'bloggersvideo15com',
       '151836595805_699672039100', 'v7webinar', 'bloggersvideo18webinar',
       'v6webinar', '151836595805_699672039103', 'bloggersvideo1webinar',
       '151836595805_699672039109', 'bloggersvideo2webinar',
       'bloggersvideo18com', 'bloggersvideo12com_at',
       '151836595805_699672039106', 'bloggersvideo19com',
       '151836595805_673801336999', 'bloggersvideo22com', 'video1com_new',
       'bloggersvideo10com', 'ntc1', 'bloggersvideo15com_python', 'b11',
 

In [ ]:
data_Deals['Content'] = data_Deals['Content'].fillna('unknown_content')

In [ ]:
#меняем формат 'Content'
data_Deals['Content'] = data_Deals['Content'].astype('category')

In [ ]:
data_Deals['Term'].unique()

array(['women', nan, '21_06_2024', 'recentlymoved', 'Com_august',
       'it career hub', 'interest_work', 'wide', 'retargeting',
       '21_05_2024', 'LAL1', 'invitation', '19_06_2024', '30_05_2024',
       'ich', '10_04_2024', 'it%20career%20hub', '18_06_2024',
       'invitation\\', '_', '07_06_2024', '14_06_2024', '13_06_2024',
       '03_06_2024', 'interest_programming_WebDev', '12_06_2024',
       '28_05_2024', 'it hub', 'itcareerhub', '31_05_2024', '05_06_2024',
       'it career hub_', '15_05_2024', 'interest_work_WebDev',
       '29_03_2024', 'lost_does_not_answer', 'айти карьер хаб',
       '23_05_2024', '13_01_2024', '1_day_before', '29_05_2024',
       '01_02_2024', '04_06_2024', 'berlin_wide', '16_05_2024',
       '22_05_2024', '19_03_2024', '23_01_2024', '27_05_2024',
       '23_04_2024', '06_04_2024', 'accountant_wide', '10_05_2024',
       '13_05_2024', '18_05_2024', '14_05_2024', 'b', '03_05_2024',
       '12_05_2024', '06_05_2024', '1_05_2024', '20_04_2024',
       '0

In [ ]:

# 1. Приведение к нижнему регистру и удаление пробелов по краям
data_Deals['Term'] = data_Deals['Term'].astype(str).str.strip().str.lower()

# 2. Удаляем подчёркивания только в начале и в конце строки
data_Deals['Term'] = data_Deals['Term'].str.replace(r'^_+|_+$', '', regex=True)

# 3. Замена ошибочных и дубль-значений
replace_map = {
    'it20career20hub': 'it_career_hub',
    'itcareerhub': 'it_career_hub',
    'it career hub': 'it_career_hub',
    'айти карьер хаб': 'it_career_hub',
    'it%20career%20hub': 'it_career_hub',
    'it_20career_20hub':'it_career_hub',
    'it hub': 'it_career_hub',
    'recentlymovedhttps://itcareerhub.de/ru/questionnaire?utm_source=facebook': 'recentlymoved',

    # Аналитика данных — сводим к единому тегу
    'курсы аналитика с нуля': 'interest_dataanalytics',
    'data analysis analytics': 'interest_dataanalytics',
    'дата аналитик курсы': 'interest_dataanalytics',
    'аналитик в it обучение': 'interest_dataanalytics',
    'аналитиквитобучение': 'interest_dataanalytics',
    'аналитика данных с нуля': 'interest_dataanalytics',
    'дата аналитика': 'interest_dataanalytics',
    'курсы аналитика': 'interest_dataanalytics',
    'аналитик данных обучение': 'interest_dataanalytics',
    'data analyst курсы': 'interest_dataanalytics',
    'курсы аналитик данных': 'interest_dataanalytics',
    'обучение на аналитика данных': 'interest_dataanalytics',
    'курсы data analyst': 'interest_dataanalytics',
    'data analyst': 'interest_dataanalytics',
    'data analyst обучение': 'interest_dataanalytics',
    'дата аналитик это': 'interest_dataanalytics',
    'курсы по дата аналитике': 'interest_dataanalytics',

    'wide_pythondeveloper': 'wide_python-developer',
    'wide_qaengineer': 'wide_qa-engineer',
    'wide_webdesigner': 'wide_webdesigner',
}

# 4. Применяем замену
data_Deals['Term'] = data_Deals['Term'].replace(replace_map)

# 5. Заполняем пустые
data_Deals['Term'] = data_Deals['Term'].replace('', 'unknown').fillna('unknown')

# 6. Проверяем результат
print(data_Deals['Term'].value_counts().head(20))

Term
nan                            9113
wide                           3666
com_august                     1525
recentlymoved                   756
women                           639
lal1                            548
retargeting                     479
invitation                      453
interest_work_webdev            310
interest_programming_webdev     258
it_career_hub                   163
b                               147
accountant_wide                 121
1_day_before                    107
1_05_2024                       105
26_01_2024                       92
20_10_2023                       91
allcomabc                        87
07_02_2024                       84
com_july_1                       83
Name: count, dtype: int64


In [ ]:
data_Deals['Term'] = data_Deals['Term'].astype('category')

In [ ]:
data_Deals['Source']= data_Deals['Source'].astype('category')


data_Deals['Source'].unique()

['Facebook Ads', 'Organic', 'Telegram posts', 'Google Ads', 'Youtube Ads', ..., 'Tiktok Ads', 'Bloggers', 'Partnership', 'Test', 'Offline']
Length: 13
Categories (13, object): ['Bloggers', 'CRM', 'Facebook Ads', 'Google Ads', ..., 'Test', 'Tiktok Ads',
                          'Webinar', 'Youtube Ads']

In [ ]:
data_Deals['Payment Type'].unique()

array([nan, 'One Payment', 'Recurring Payments', 'Reservation'],
      dtype=object)

In [ ]:
# Заполняем пропуски
data_Deals['Payment Type'] = data_Deals['Payment Type'].fillna('no payment')



In [ ]:
data_Deals['Product'].unique()

array([nan, 'Web Developer', 'Digital Marketing', 'UX/UI Design',
       'Find yourself in IT', 'Data Analytics'], dtype=object)

In [ ]:
# Заполняем пропуски значением "No Product"
data_Deals['Product'] = data_Deals['Product'].fillna('No Cours')

# Приводим к категориальному типу
data_Deals['Product'] = data_Deals['Product'].astype('category')

In [ ]:
data_Deals['Education Type'].unique()

array([nan, 'Morning', 'Evening'], dtype=object)

In [ ]:
# Заполняем пропуски значением "No Product"
data_Deals['Education Type'] = data_Deals['Education Type'].fillna('No Cours')

# Приводим к категориальному типу
data_Deals['Education Type'] = data_Deals['Education Type'].astype('category')

In [ ]:
# Заполняем пропуски значением "No Product"
data_Deals['Course duration'] = data_Deals['Course duration'].fillna('0')

# Приводим к категориальному типу
data_Deals['Course duration'] = data_Deals['Course duration'].astype(int)

In [ ]:
# Преобразуем 'Created Time' в формат datetime
data_Deals['Created Time'] = pd.to_datetime(data_Deals['Created Time'], errors='coerce')

/tmp/ipython-input-3400758450.py:2: UserWarning: Parsing dates in %d.%m.%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  data_Deals['Created Time'] = pd.to_datetime(data_Deals['Created Time'], errors='coerce')


In [ ]:
# Заполняем пропуски значением "0"
data_Deals['Months of study'] = data_Deals['Months of study'].fillna('0')

# Приводим к int типу
data_Deals['Months of study'] = data_Deals['Months of study'].astype(int)

In [ ]:
data_Deals['Initial Amount Paid'].unique()

array([nan, 0, 1000, '€ 3.500,00', 500, 100, 4500, 300, 200, 2000, 11000,
       4000, 3000, 3500, 11500, 1200, 1500, 1, 5000, 600, 700, 350, 9,
       400, 450], dtype=object)

In [ ]:
import pandas as pd

# Преобразуем в строки и убираем € и пробелы
data_Deals['Initial Amount Paid'] = (
    data_Deals['Initial Amount Paid']
    .astype(str)
    .str.replace('€', '', regex=False)
    .str.replace(' ', '', regex=False)
    .str.replace(',', '.', regex=False)   # сначала заменяем запятые на точки
    .str.replace(r'(?<=\d)\.(?=\d{3}\b)', '', regex=True)  # удаляем точки, если они разделяют тысячи
)

# Преобразуем к числу
data_Deals['Initial Amount Paid'] = pd.to_numeric(
    data_Deals['Initial Amount Paid'], errors='coerce'
)

# Заполняем пропуски нулями
data_Deals['Initial Amount Paid'] = data_Deals['Initial Amount Paid'].fillna(0)

# Приводим к int, если нет дробей
if (data_Deals['Initial Amount Paid'] % 1 == 0).all():
    data_Deals['Initial Amount Paid'] = data_Deals['Initial Amount Paid'].astype(int)

# Проверим результат
print(data_Deals['Initial Amount Paid'].unique())


[    0  1000  3500   500   100  4500   300   200  2000 11000  4000  3000
 11500  1200  1500     1  5000   600   700   350     9   400   450]


In [ ]:
data_Deals['Offer Total Amount'].unique()

array([nan, 2000, 9000, 11000, 3500, 4500, '€ 2.900,00', 6500, 4000, 3000,
       10000, 2500, 5000, 11500, 1, 1000, 1200, 0, 1500, '€ 11398,00',
       11111, 6000], dtype=object)

In [ ]:
# Очистка и преобразование Offer Total Amount
data_Deals['Offer Total Amount'] = (
    data_Deals['Offer Total Amount']
    .astype(str)
    .str.replace('€', '', regex=False)
    .str.replace(' ', '', regex=False)
    .str.replace(',', '.', regex=False)
    .str.replace(r'(?<=\d)\.(?=\d{3}\b)', '', regex=True)
)

# Переводим в числа
data_Deals['Offer Total Amount'] = pd.to_numeric(
    data_Deals['Offer Total Amount'], errors='coerce'
)

# Заменяем NaN на 0
data_Deals['Offer Total Amount'] = data_Deals['Offer Total Amount'].fillna(0)

# Если все значения целые — приводим к int
if (data_Deals['Offer Total Amount'] % 1 == 0).all():
    data_Deals['Offer Total Amount'] = data_Deals['Offer Total Amount'].astype(int)

# Проверяем результат
print(data_Deals['Offer Total Amount'].unique())


[    0  2000  9000 11000  3500  4500  2900  6500  4000  3000 10000  2500
  5000 11500     1  1000  1200  1500 11398 11111  6000]


In [ ]:
# все закрытые сделки переводим в 'Payment Done'
data_Deals['Payment Type'] = data_Deals.get('Payment Type', np.nan)
data_Deals['Stage'] = data_Deals.get('Stage', np.nan)

# Условие 1: Offer Total Amount = Initial Amount Paid
mask1 = (data_Deals['Offer Total Amount'] == data_Deals['Initial Amount Paid']) & (data_Deals['Offer Total Amount'] > 0)
mask2 = (data_Deals['Offer Total Amount'] > data_Deals['Initial Amount Paid']) & (data_Deals['Initial Amount Paid'] > 0)

# Применяем условия к Pay Type
data_Deals.loc[mask1, 'Payment Type'] = 'One Payment'
data_Deals.loc[mask2, 'Payment Type'] = 'Partial Pay'

# Применяем условия к Stage
data_Deals.loc[mask1, 'Stage'] = 'Payment Done'
data_Deals.loc[mask2, 'Stage'] = 'Payment Done'

# Проверим результат
data_Deals[['Offer Total Amount', 'Initial Amount Paid', 'Payment Type', 'Stage']].head(10)

,Offer Total Amount,Initial Amount Paid,Payment Type,Stage
0,0,0,no payment,New Lead
1,2000,0,no payment,New Lead
2,0,0,no payment,Lost
3,0,0,no payment,Lost
4,0,0,no payment,Lost
5,0,0,no payment,Need a consultation
6,0,0,no payment,Need To Call
7,0,0,no payment,Need a consultation
8,0,0,no payment,Lost
9,0,0,no payment,Lost


In [ ]:
# Приводим к категориальному типу
data_Deals['Payment Type'] = data_Deals['Payment Type'].astype('category')

In [ ]:
# смотрим уникальные
data_Deals['Contact Name'].isna().sum()

np.int64(60)

In [ ]:
#удаляем пустые строки
data_Deals = data_Deals.dropna(subset=['Contact Name'])

In [ ]:
data_Deals['City'].unique()

array([nan, 'Crailsheim', 'Dortmund', 'Stuttgart', 'München', 'Berlin',
       'Wien', 'Offenbach am Main', 'Eberbach', 'Görlitz', 'Pfedelbach',
       'Unterhaching',
       'Karl-Liebknecht str. 24, Hildburghausen, Thüringen',
       'Rüdesheim am Rhein', 'Dresden', 'Gummersbach', '-', 'Kassel',
       'Wenzenbach', 'Merseburg', 'Gommern', 'Pommelsbrunn', 'Duisburg',
       'Herzogenrath', 'Schwandorf', 'Mainz', 'Podskalie', 'Zinnowitz',
       'Quedlinburg', 'Poland , Gdansk , Al. Grunwaldzka 7, ap. 1a',
       'Wolfsburg', 'Weilburg', 'Dillenburg', 'Neu-Ulm',
       'Lauter-Bernsbach', 'Bonn', 'Riedstadt', 'Rosenheim',
       'Mönchengladbach', 'Neuburg', 'Rostock', 'Bad Oeynhausen',
       'Chemnitz', 'Diez', 'Nürnberg', 'Laubach', 'Düren', 'Düsseldorf',
       'Zwickau', 'Bremen', 'Halle', 'Erbach', 'Jünkerath', 'Magdeburg',
       'Celle', 'Germering', 'Kleve', 'Leinfelden-Echterdingen',
       'Garmisch-Partenkirchen', 'Leipzig', 'Hof', 'Lünen', 'Murr',
       'Bochum', 'Leonbe

In [ ]:
# Словарь для очистки
STREET_TOKENS = {
    'str', 'str.', 'straße', 'strasse', 'street', 'weg', 'allee', 'al', 'al.',
    'platz', 'pl.', 'ap', 'apt', 'haus', 'nr', 'no', '№', 'building', 'road'
}

REGION_TOKENS = {
    'germany', 'deutschland', 'österreich', 'austria', 'poland', 'polska',
    'bayern', 'sachsen', 'nrw', 'rheinland-pfalz', 'baden-württemberg',
    'hessen', 'niedersachsen', 'saarland', 'schleswig-holstein', 'brandenburg'
}

MANUAL_MAP = {
    'muenchen': 'München',
    'munchen': 'München',
    'koeln': 'Köln',
    'duesseldorf': 'Düsseldorf',
    'nuernberg': 'Nürnberg',
    'weisswasser': 'Weißwasser',
    'helmstidde': 'Helmstedt',
    'saarbrucken': 'Saarbrücken',
    'villingen-schwenningen': 'Villingen-Schwenningen',
    'rheda-wiedenbruck': 'Rheda-Wiedenbrück',
    'wurzburg': 'Würzburg',
    'gdansk': 'Gdańsk',
    'slubice': 'Słubice',
    'gdynia': 'Gdynia',
    'szczecin': 'Szczecin',
    'warszawa': 'Warszawa',
    'moscow': 'Moscow',
    'saint petersburg': 'Saint Petersburg',
    'minsk': 'Minsk',
    'belgrade': 'Belgrade',
    'kharkiv': 'Kharkiv',
    'dubai': 'Dubai',
    'almaty': 'Almaty',
    'phuket': 'Phuket',
    'baden-baden': 'Baden-Baden',
}

LOWER_CONNECTORS = {'am', 'im', 'an', 'bei', 'der', 'den', 'vom', 'von', 'und', 'zum', 'zur', 'an der'}

In [ ]:
# Функции для очистки
def normalize_basic(s: str) -> str:
    """Убираем лишние пробелы, спецсимволы, приводим дефисы к одному виду."""
    s = str(s)
    s = s.replace('–', '-').replace('—', '-').replace('\u2011', '-')  # разные типы дефисов
    s = s.replace('\xa0', ' ')  # неразрывные пробелы
    s = s.strip().strip('"').strip("'")
    s = re.sub(r'\s+', ' ', s)
    return s

def token_is_city(tok: str) -> bool:
    """Определяет, похоже ли слово на город."""
    t = tok.strip().lower()
    if not t or any(ch.isdigit() for ch in t):
        return False
    t_clean = t.replace('.', '')
    if t_clean in REGION_TOKENS or t_clean in STREET_TOKENS:
        return False
    if t in {'-', '_'}:
        return False
    return bool(re.search(r'[a-zA-Zа-яА-ЯäöüÄÖÜßłńóśźąęćż]', t))

def pick_city_from_address(s: str) -> str | None:
    """Если строка с запятыми — выбираем наиболее похожий на город фрагмент."""
    tokens = [normalize_basic(t) for t in s.split(',')]
    for tok in reversed(tokens):
        if token_is_city(tok):
            return tok
    for tok in tokens:
        if token_is_city(tok):
            return tok
    return None

def smart_title(city: str) -> str:
    """Title-case с сохранением строчных предлогов."""
    def fix_word(w):
        return w if w in LOWER_CONNECTORS else w.capitalize()
    parts = []
    for p in city.split('-'):
        words = [fix_word(w) for w in p.split()]
        parts.append(' '.join(words))
    return '-'.join(parts)

def clean_city(value):
    """Главная функция очистки"""
    if pd.isna(value):
        return np.nan
    s = str(value).strip()
    if s in {'', '-', '_'}:
        return np.nan

    s = normalize_basic(s)
    # Удаляем всё в скобках
    s = re.sub(r'\(.*?\)', '', s).strip()

    # Извлекаем город из адресной строки
    if ',' in s:
        candidate = pick_city_from_address(s)
        if candidate:
            s = candidate
        # Удаляем строки, содержащие явные адресные элементы
    if any(tok in s.lower().replace('.', '') for tok in STREET_TOKENS) and any(ch.isdigit() for ch in s):
        return np.nan

    s_lower = s.lower()
    s_lower = re.sub(r'\s*-\s*', '-', s_lower)
    s_lower = re.sub(r'\s+', ' ', s_lower).strip()

    # Если это страна/регион
    if s_lower in REGION_TOKENS:
        return np.nan

    # Ручные маппинги
    if s_lower in MANUAL_MAP:
        return MANUAL_MAP[s_lower]

    cleaned = smart_title(s_lower)
    return cleaned

In [ ]:
data_Deals['City'] = data_Deals['City'].apply(clean_city)
data_Deals['City'].unique()

array([nan, 'Crailsheim', 'Dortmund', 'Stuttgart', 'München', 'Berlin',
       'Wien', 'Offenbach am Main', 'Eberbach', 'Görlitz', 'Pfedelbach',
       'Unterhaching', 'Thüringen', 'Rüdesheim am Rhein', 'Dresden',
       'Gummersbach', 'Kassel', 'Wenzenbach', 'Merseburg', 'Gommern',
       'Pommelsbrunn', 'Duisburg', 'Herzogenrath', 'Schwandorf', 'Mainz',
       'Podskalie', 'Zinnowitz', 'Quedlinburg', 'Gdańsk', 'Wolfsburg',
       'Weilburg', 'Dillenburg', 'Neu-Ulm', 'Lauter-Bernsbach', 'Bonn',
       'Riedstadt', 'Rosenheim', 'Mönchengladbach', 'Neuburg', 'Rostock',
       'Bad Oeynhausen', 'Chemnitz', 'Diez', 'Nürnberg', 'Laubach',
       'Düren', 'Düsseldorf', 'Zwickau', 'Bremen', 'Halle', 'Erbach',
       'Jünkerath', 'Magdeburg', 'Celle', 'Germering', 'Kleve',
       'Leinfelden-Echterdingen', 'Garmisch-Partenkirchen', 'Leipzig',
       'Hof', 'Lünen', 'Murr', 'Bochum', 'Leonberg',
       'Bad Homburg Vor der Höhe', 'Kiel', 'Theres', 'Lüchow', 'Stolberg',
       'Bautzen', 'Weide

In [ ]:
# Заменяем пустые значения на "unknown_city"
data_Deals['City'] = data_Deals['City'].fillna('unknown')


In [ ]:
# Приводим к категориальному типу
data_Deals['City'] = data_Deals['City'].astype('category')

In [ ]:
data_Deals['Contact Name']= data_Deals['Contact Name'].fillna('unknown')

data_Deals['Contact Name'] = data_Deals['Contact Name'].astype('category')

In [ ]:
data_Deals['Level of Deutsch'].unique()

array([nan, 'в1', 'A2', 'б1', 'b1', 'B1', 'в1-в2', 'B2', 'C2', 'с1', 'Б1',
       'а2', 'а1', 'а0', 'б2', 'Б2', 'В1', 'А2',
       'B1 будет в феврале 2025', 'Detmold, Paulinenstraße 95, 32756',
       'Сам оценивает на B2, 13 лет живет в Германии', 'в2', 'В1-В2',
       'Б1 ( ждет Б2)', 'А2-В1',
       'lэкзамен - 6 июля на В1. курсы вечером (но уверенно говорит на B1)',
       'Гражданка Германии уже год в Германии Учит немецкий и в сентябре b1 через гос-во проходит, а не через ДЖЦ, вечером учится 3 р в неделю с 18 до 21',
       '-', 'А2 ( Б1 в июне)', 'B1 в процессе обучения',
       'ЯЗ: нем В1 был экз 03.05 повтор и сейчас ждет результаты. Технический англ был. А1 сейчас. ОБР: 2 во информационные и комп сети - инженер системоте',
       'В1 в сентябре', 'Нет', 'С1', 0, 'Ждем B1',
       'А1 сертиф, но по факту А2', 'a2', 'Пока А2, сдает 17 05 B1',
       'окончание 13.06 курса на b1', 'A1', 'b2',
       'Thorn-Prikker-Str. 30, Hagen, 58093', 'В2',
       'нулевой уровень, только 

In [ ]:
def clean_german_level(value):
    if pd.isna(value):
        return None

    # Приводим к строке и убираем пробелы
    text = str(value).strip().lower()


    text = (
        text.replace('а', 'a')
            .replace('б', 'b')
            .replace('в', 'b')
            .replace('с', 'c')
            .replace('с1', 'c1')
    )

    # Находим все уровни A0–C2
    match = re.findall(r'\b[abc][0-2]\b', text)

    if not match:
        return None

    # Берем наивысший уровень (если несколько)
    # порядок уровней для сортировки
    level_order = {'a0': 0, 'a1': 1, 'a2': 2, 'b1': 3, 'b2': 4, 'c1': 5, 'c2': 6}
    highest = sorted(match, key=lambda x: level_order[x])[ -1 ].upper()

    return highest

# Применяем к колонке
data_Deals['Level of Deutsch'] = data_Deals['Level of Deutsch'].apply(clean_german_level)

# Заменяем на None пропущенные значения
data_Deals['Level of Deutsch'] = data_Deals['Level of Deutsch'].fillna('unknown')



print(data_Deals['Level of Deutsch'].unique())


['unknown' 'B1' 'A2' 'B2' 'C2' 'C1' 'A1' 'A0']


In [ ]:
#заполняем пустые значения
data_Deals['Level of Deutsch'] = data_Deals['Level of Deutsch'].fillna('unknown')

# Приводим к категориальному типу
data_Deals['Level of Deutsch'] = data_Deals['Level of Deutsch'].astype('category')

In [ ]:
data_Deals['Contact Name']= data_Deals['Contact Name'].replace('unknown', np.nan).astype('category')


In [ ]:
# Преобразуем в timedelta, ошибки превратятся в NaT
data_Deals['SLA'] = pd.to_timedelta(data_Deals['SLA'], errors='coerce')

#заполним пустые значения
data_Deals['SLA'] = data_Deals['SLA'].fillna(pd.Timedelta(seconds=0))

In [ ]:
data_Deals.dtypes

,0
Id,int64
Deal Owner Name,category
Closing Date,datetime64[ns]
Quality,int64
Stage,category
Lost Reason,category
Page,category
Campaign,category
SLA,timedelta64[ns]
Content,category


In [ ]:
data_Deals.isna().sum()

,0
Id,0
Deal Owner Name,0
Closing Date,6918
Quality,0
Stage,0
Lost Reason,0
Page,0
Campaign,0
SLA,0
Content,0


In [ ]:
#сохраняем чистый датасет
data_Deals.to_excel('/content/drive/MyDrive/Projekt_PythonDA/data_Deals_cleaned.xlsx')





In [ ]:
# Преобразуем в timedelta, ошибки превратятся в NaT
#data_Deals['SLA'] = pd.to_timedelta(data_Deals['SLA'], errors='coerce')